# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

Voilà la structure du body de la page d'accueil :
<body>
	<div class="container">
		<div class="row header-box">...</div> -- Quotes to Scrape   +   Login
		<div class="row">
			<div class="col-md-8">
				<div class="quote" itemscope itemtype="http://schema.org/CreativeWork"> -- boîte de la première citation de la page
					<span class="text" itemprop="text">...texte de la citation...</span>
					<span>
						"by "
						<small class="author" itemprop="author">...nom de l'auteur...</small>
						<a href="/author/...nom auteur...">(about)</a>
					</span>
					<div class="tags">
						" Tags: "
						<meta class="keywords" itemprop="keywords" content="...liste mots-clés séparés par une , ...">
						<a class="tag" href="/tag/...nom tag.../page/1/">...nom tag...</a>
						...
						<a class="tag" href="/tag/...nom tag.../page/1/">...nom tag...</a>
					</div>
				</div>
				<div class="quote" itemscope itemtype="http://schema.org/CreativeWork">...</div> -- boîte suivante
				...
				<div class="quote" itemscope itemtype="http://schema.org/CreativeWork">...</div> -- dernière boîte de la page
				<nav>
					<ul class="pager">
						<li class="previous">
							<a href="/page/...n° page précédente.../">
								<span aria-hidden="true"><-</span>
								" Previous"
							</a>
						</li>
						<li class="next">
							<a href="/page/...n° page suivante.../">
								"Next "
								<span aria-hidden="true">-></span>
							</a>
						</li>
					</ul>
				</nav>					
			</div>
			<div class="col-md-4 tags-box">...</div> -- Top Ten tags
		</div>
	</div>
	<footer class="footer">...</footer> -- Quotes by: GoodReads.com  +  Made with ❤ by Zyte
</body>

Bien entendu, la fiche d'un auteur n'a pas besoin d'être téléchargée plusieurs fois, son URL est toujours la même.
Les emboîtements de tags ne sont utiles que pour la présentation de la page web, chaque information d'une citation doit être cherchée à son niveau quel qu'il soit au sein de la div de classe quote.
Les guillemets typographiques ne sont pas nécessaire pour l'enregistrement de la citation, c'est juste un artifice de présentation au sein de la page web.

In [1]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [2]:
ENTETES = {"User-Agent": "TPFormation/1.0 (fre.per@univ-lyon1.fr) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (fre.per@univ-lyon1.fr) python-requests'}


In [3]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (fre.per@univ-lyon1.fr) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [3]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [8]:
regles = lire_robots(SITE)
# TODO : calculer autorise_pages et autorise_auteur (si regles vaut None, aucune restriction annoncée)
autorise_pages = True if regles is None else regles.can_fetch(ENTETES['User-Agent'], urljoin(SITE, "/page/2/"))
autorise_auteur = True if regles is None else regles.can_fetch(ENTETES['User-Agent'], urljoin(SITE, "/auteur/Albert-Einstein"))

https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée


In [9]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [12]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
nb_requetes = 0
def extraire_citations(url):
    citations = []
    global nb_requetes
    while url:
        url_page = url
        reponse = get_poli(url)
        nb_requetes += 1
        if reponse.status_code != 200:
            raise RuntimeError("Échec de la requête : " + url)
        page = BeautifulSoup(reponse.text, 'html.parser')
        for bloc in page.select('.quote'):
            texte = bloc.select_one('.text').get_text(strip=True)[1:-1]  # on enlève les guillemets
            auteur = bloc.select_one('.author').get_text(strip=True)
            tags = [tag.get_text(strip=True) for tag in bloc.select('.tags .tag')]
            url_auteur = urljoin(SITE, bloc.select_one('a').get('href'))
            citations.append((texte, auteur, tags, url_page, url_auteur))
        lien_suivant = page.select_one('li.next > a')
        url = urljoin(SITE, lien_suivant['href']) if lien_suivant else None
    return citations
citations = extraire_citations(SITE)
citations = pd.DataFrame(citations, columns=['texte', 'auteur', 'tags', 'url_page', 'url_auteur'])

In [5]:
citations.count()

texte         100
auteur        100
tags          100
url_page      100
url_auteur    100
dtype: int64

In [13]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [14]:
# À vous : collecter une fiche par auteur (variable attendue : auteurs)
auteurs = []
for url_auteur in citations.url_auteur.unique():
    reponse = get_poli(url_auteur)
    nb_requetes += 1
    if reponse.status_code != 200:
        raise RuntimeError("Échec de la requête : " + url_auteur)
    page = BeautifulSoup(reponse.text, 'html.parser')
    auteur = page.select_one('h3.author-title').get_text(strip=True)
    date_naissance = page.select_one('span.author-born-date').get_text(strip=True)
    lieu_naissance = page.select_one('span.author-born-location').get_text(strip=True)[3:]  # on enlève le "in "
    biographie = page.select_one('div.author-description').get_text(strip=True)
    auteurs.append((url_auteur, auteur, date_naissance, lieu_naissance, biographie))
auteurs = pd.DataFrame(auteurs, columns=['url_auteur', 'auteur', 'date_naissance', 'lieu_naissance', 'biographie'])

In [15]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 60 requêtes pour 60 attendues au minimum : fiches non redemandées


In [16]:
auteurs.head()

,url_auteur,auteur,date_naissance,lieu_naissance,biographie
0,https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,"March 14, 1879","Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,https://quotes.toscrape.com/author/J-K-Rowling,J.K. Rowling,"July 31, 1965","Yate, South Gloucestershire, England, The Unit...",See also: Robert GalbraithAlthough she writes ...
2,https://quotes.toscrape.com/author/Jane-Austen,Jane Austen,"December 16, 1775","Steventon Rectory, Hampshire, The United Kingdom",Jane Austen was an English novelist whose work...
3,https://quotes.toscrape.com/author/Marilyn-Monroe,Marilyn Monroe,"June 01, 1926",The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...
4,https://quotes.toscrape.com/author/Andre-Gide,André Gide,"November 22, 1869","Paris, France",André Paul Guillaume Gide was a French author ...


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [25]:
# À vous : citations_completes, top_auteurs, top_tags
citations_completes = pd.merge(citations.drop(columns=["auteur"]), auteurs, on='url_auteur', how='left')
top_auteurs = citations_completes.groupby('auteur').size().sort_values(ascending=False).head(5)
citations_tags = citations_completes.explode('tags')
top_tags = citations_tags.groupby('tags').size().sort_values(ascending=False).head(10)

In [26]:
citations_completes.head(2)

,texte,tags,url_page,url_auteur,auteur,date_naissance,lieu_naissance,biographie
0,The world as we have created it is a process o...,"[change, deep-thoughts, thinking, world]",https://quotes.toscrape.com,https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,"March 14, 1879","Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,"It is our choices, Harry, that show what we tr...","[abilities, choices]",https://quotes.toscrape.com,https://quotes.toscrape.com/author/J-K-Rowling,J.K. Rowling,"July 31, 1965","Yate, South Gloucestershire, England, The Unit...",See also: Robert GalbraithAlthough she writes ...


In [27]:
top_auteurs

auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Mark Twain          6
Dr. Seuss           6
dtype: int64

In [30]:
top_tags

tags
love                    14
inspirational           13
life                    13
humor                   12
books                   11
reading                  7
friendship               5
friends                  4
truth                    4
attributed-no-source     3
dtype: int64

In [29]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
2. Ce qui était **différent**.
3. Ce que vous avez dû **changer** dans votre démarche.
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.

> 

Dans les deux cas :
* il n'y a pas de robot.txt
* il faut naviguer sur plusieurs pages (next) pour avoir la liste complète des objets à scraper, chaque page contient l'url de sa page suivante
* les informations plus détaillées sur l'objet sont dans une autre page accessible par une url affichée dans la page initiale
* il n'y a pas de javascript qui affiche des informations non visibles dans le contenu html
* une caractéristique (catégorie côté livres, tag côté citations) fait l'objet d'une page dédiée qui filtre tous les objets partageant cette caractéristique

Il y a quelques différences :
* il y a autant de pages de description détaillée que de livres, en revanche il n'y a qu'une page par auteur, donc moins que de citations
* côté citations, le champ tags contient une liste, on n'a pas de champ avec des listes côté livres

Côté citation, il a donc fallu lire les pages auteurs séparément de la lecture des pages citations pour ne pas encombrer le site par des requêtes redondantes, et déplier la liste des tags en créant une ligne par tag (fonction explode).

Bien entendu, si le site changeait de contenu ou de style (noms des classes) ou utilisait du javascript pour afficher des données, le script ne serait plus fonctionnel. Ce dernier est très dépendant du layout de la page, ce qui en fait sa fragilité. Si robot.txt apparaissait, il faudrait également revoir la conformité de ce script.
